Purpose: Run Fisher's exact tests for enrichment of CAM genes in maSigPro results (per genotype and per ZT-peak group).<br>
Author: Anna Pardo<br>
Date initiated: Dec. 8, 2025

In [1]:
# load modules
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import statistics
import scipy.stats as stats
import numpy as np
from statsmodels.stats.multitest import fdrcorrection
from venn import venn
from matplotlib.patches import Patch
import json

In [2]:
# load time-structured gene information
cinfo = pd.read_csv("./clusters_ngenes_gt_ztgroup_with36-61-43.txt",sep="\t",header="infer")
cinfo.head()

,cluster,genotype,n_CAM_genes,n_all_genes,gtc,ZTpeak
0,1,53,0.0,319,53_1,5.0
1,2,53,1.0,656,53_2,21.0
2,3,53,1.0,882,53_3,17.0
3,4,53,0.0,1071,53_4,9.0
4,5,53,0.0,428,53_5,21.0


In [3]:
cinfo["genotype"].unique()

array(['53', 'Eudy', '13', '52', '48', 'G', '19', '46', '56', '45', '18',
       '70', '1AB', '55', '51', '2AB', '37', 'all', '36', '61', '43'],
      dtype=object)

In [4]:
# remove 'all'
cgt = cinfo[cinfo["genotype"]!="all"]

In [6]:
# load cluster membership info
## annotate whether each gene is CAM (Y/N)
## for which I also need the CAM annotation
cam = pd.read_csv("../degs_downstream/camgenes_Ya_Yf_orthology_synteny.txt",sep="\t",header="infer")

In [11]:
camids = list(cam["GeneID"].unique())

In [35]:
# define a function to load & summarize the data for a given genotype
def load_sum_gt(gt):
    if gt in ["all","gt37"]:
        if gt=="all":
            gtc = pd.read_csv("./hck6_treatonly_"+gt+"_18-Nov.tsv",sep="\t",header="infer")
        else:
            gtc = pd.read_csv("./hck6_treatonly_"+gt+"_17-Nov.tsv",sep="\t",header="infer")
            gt = "37"
    else:
        gtc = pd.read_csv("./Yg"+gt+"_hclust_k6_clusters.txt",sep="\t",header="infer")
        
    gtc["genotype"] = gt
    # set up a CAM column
    camornot = []
    for i in gtc["GeneID"]:
        if i in camids:
            camornot.append("Y")
        else:
            camornot.append("N")
    gtc["CAM_notCAM"] = camornot
    
    # set up a subgenome column
    sg = []
    for i in gtc["GeneID"]:
        if i.startswith("Yucal"):
            sg.append("Ya")
        else:
            sg.append("Yf")
    gtc["subgenome"] = sg
    
    return gtc

In [36]:
dflist = []
for f in os.listdir("./"):
    if(f.endswith("clusters.txt") and f.startswith("Yg")):
        print(f)
        gt = f.split("_")[0].lstrip("Yg")
        print(gt)
        dflist.append(load_sum_gt(gt))

YgG_hclust_k6_clusters.txt
G
YgEudy_hclust_k6_clusters.txt
Eudy
Yg36_hclust_k6_clusters.txt
36
Yg43_hclust_k6_clusters.txt
43
Yg13_hclust_k6_clusters.txt
13
Yg45_hclust_k6_clusters.txt
45
Yg19_hclust_k6_clusters.txt
19
Yg51_hclust_k6_clusters.txt
51
Yg2AB_hclust_k6_clusters.txt
2AB
Yg52_hclust_k6_clusters.txt
52
Yg46_hclust_k6_clusters.txt
46
Yg70_hclust_k6_clusters.txt
70
Yg48_hclust_k6_clusters.txt
48
Yg55_hclust_k6_clusters.txt
55
Yg18_hclust_k6_clusters.txt
18
Yg61_hclust_k6_clusters.txt
61
Yg53_hclust_k6_clusters.txt
53
Yg56_hclust_k6_clusters.txt
56
Yg1AB_hclust_k6_clusters.txt
1AB


In [37]:
dflist.append(load_sum_gt("gt37"))

In [38]:
allclusters = pd.concat(dflist)

In [39]:
allclusters["genotype"].unique()

array(['G', 'Eudy', '36', '43', '13', '45', '19', '51', '2AB', '52', '46',
       '70', '48', '55', '18', '61', '53', '56', '1AB', '37'],
      dtype=object)

In [40]:
set(allclusters["genotype"].unique()).difference(set(cgt["genotype"].unique()))

set()

In [41]:
allclusters.head()

,cluster,GeneID,genotype,CAM_notCAM,subgenome
0,1,Yucal.01G003400.v2.1,G,N,Ya
1,1,Yucal.01G004800.v2.1,G,N,Ya
2,2,Yucal.01G004900.v2.1,G,N,Ya
3,3,Yucal.01G005300.v2.1,G,N,Ya
4,4,Yucal.01G012100.v2.1,G,N,Ya


# Run enrichment of CAM genes for each cluster
(both for each subgenome individually and both together)<br>
also: enrichment of subgenomes in clusters

In [42]:
# create some columns indicating cluster_genotype, cluster_subgenome, genotype_subgenome, and cluster_genotype_subgenome
allclusters["genotype_cluster"] = allclusters["genotype"]+"_"+allclusters["cluster"].astype(str)
allclusters["genotype_subgenome"] = allclusters["genotype"]+"_"+allclusters["subgenome"]
allclusters["genotype_cluster_subgenome"] = allclusters["genotype_cluster"]+"_"+allclusters["subgenome"]
allclusters["cluster_subgenome"] = allclusters["cluster"].astype(str)+"_"+allclusters["subgenome"]

In [43]:
allclusters.head()

,cluster,GeneID,genotype,CAM_notCAM,subgenome,genotype_cluster,genotype_subgenome,genotype_cluster_subgenome,cluster_subgenome
0,1,Yucal.01G003400.v2.1,G,N,Ya,G_1,G_Ya,G_1_Ya,1_Ya
1,1,Yucal.01G004800.v2.1,G,N,Ya,G_1,G_Ya,G_1_Ya,1_Ya
2,2,Yucal.01G004900.v2.1,G,N,Ya,G_2,G_Ya,G_2_Ya,2_Ya
3,3,Yucal.01G005300.v2.1,G,N,Ya,G_3,G_Ya,G_3_Ya,3_Ya
4,4,Yucal.01G012100.v2.1,G,N,Ya,G_4,G_Ya,G_4_Ya,4_Ya


In [45]:
# load TPM data
mdtpm = pd.read_csv("../TPM/Yg_toYgIS_allTPM_correctedmd_over1mil.txt",sep="\t",header="infer")
mdtpm.head()

/tmp/ipykernel_140/3237519056.py:2: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  mdtpm = pd.read_csv("../TPM/Yg_toYgIS_allTPM_correctedmd_over1mil.txt",sep="\t",header="infer")


,sample_name,genotype,time,treat,ZT,species,Yucal.01G000100.v2.1,Yucal.01G000200.v2.1,Yucal.01G000300.v2.1,Yucal.01G000400.v2.1,...,YufilH1095122m.g,YufilH1095123m.g,YufilH1095125m.g,YufilH1095126m.g,YufilH1095128m.g,YufilH1095131m.g,YufilH1095132m.g,YufilH1095134m.g,YufilH1095146m.g,YufilH1095147m.g
0,Y1,18,1.0,W,1.0,gloriosa,34.002815,4.546167,0.0,17.236119,...,0.000000,6.167441,1.435253,0.279077,11.420139,0.662252,1.886709,6.675105,0.0,0.000000
1,Y10,2AB,1.5,W,3.0,gloriosa,40.070758,3.628454,0.0,14.918115,...,0.713535,2.197522,15.695904,1.193256,10.172780,0.000000,2.214483,12.125756,0.0,0.000000
2,Y100,2AB,6.5,W,23.0,gloriosa,47.402599,5.201760,0.0,17.406497,...,0.314746,2.261806,21.742515,1.798380,10.256681,1.748663,2.075757,25.215640,0.0,1.838780
3,Y101,2AB,1.0,D,1.0,gloriosa,57.062380,6.374324,0.0,10.567561,...,0.000000,1.072366,27.573939,1.164591,9.105769,1.257431,2.431447,4.508369,0.0,0.813682
4,Y103,2AB,1.0,D,1.0,gloriosa,34.679279,6.087451,0.0,11.115252,...,0.000000,0.914550,28.621412,0.869052,7.076224,0.515567,2.419222,4.625880,0.0,0.867418


In [131]:
mdtpm["genotype"] = mdtpm["genotype"].astype(str)
mdtpm["genotype"].unique()

array(['18', '2AB', '1AB', '19', '15', 'Eudy', 'G', '56', '36', '13',
       '45', '52', '43', '37', '48', '55', '70', '61', '51', '46', '53',
       '16', '6', '12', '20', '50'], dtype=object)

In [133]:
# subset by genotype (actually used in maSigPro) and ZT (every 4 hours)
zttpm = mdtpm[mdtpm["ZT"].isin([1.0,5.0,9.0,13.0,17.0,21.0])]
zttpm = zttpm[~zttpm["genotype"].isin(["15","6","16","20","12","50"])]

In [134]:
# wrangle tpm data
ttpm = zttpm.set_index("sample_name").drop(["genotype","time","treat","ZT","species"],axis=1).transpose().reset_index().rename(columns={"index":"GeneID"})
ttpm.head()

sample_name,GeneID,Y1,Y101,Y103,Y104,Y105,Y111,Y114,Y116,Y118,...,SRR30854480,SRR30854481,SRR30854482,SRR30854483,SRR30854484,SRR30854485,SRR30854486,SRR30854487,SRR30854488,Y220
0,Yucal.01G000100.v2.1,34.002815,57.062380,34.679279,54.385723,47.391406,48.380647,29.091548,48.429106,35.309049,...,29.891980,27.783417,26.418632,8.504887,20.520463,24.736257,31.956674,26.805637,19.047141,39.036834
1,Yucal.01G000200.v2.1,4.546167,6.374324,6.087451,5.745473,8.458668,6.981615,3.380652,0.853778,3.546558,...,8.703255,9.056956,10.153252,5.971645,11.111843,9.005069,11.949618,10.381398,10.274897,3.534831
2,Yucal.01G000300.v2.1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.165487
3,Yucal.01G000400.v2.1,17.236119,10.567561,11.115252,17.707487,15.536940,19.211534,14.006046,18.419335,7.943992,...,16.551473,16.905682,14.693080,8.349755,13.195929,21.114249,15.557701,11.593198,13.047818,7.751035
4,Yucal.01G000500.v2.1,3.410517,3.129243,3.199743,1.184130,2.099474,3.029870,3.715974,3.284619,1.989776,...,2.349671,2.758449,3.270031,1.936751,3.382665,2.365732,3.914654,3.487815,3.345973,3.059789


In [135]:
yatpm = ttpm[ttpm["GeneID"].isin([i for i in list(ttpm["GeneID"].unique()) if i.startswith("Yucal")])]
yftpm = ttpm[ttpm["GeneID"].isin([i for i in list(ttpm["GeneID"].unique()) if i.startswith("Yufil")])]

In [47]:
# set up some useful functions for Fisher's exact test
## Fisher table setup: isPathway, isGroup
def data_setup(pathwaygenes,groupgenes,path,detype,tpmdf=ttpm):
    pathornot = []
    groupornot = []
    for g in list(tpmdf["GeneID"].unique()):
        if g in pathwaygenes:
            pathornot.append("Yes")
        else:
            pathornot.append("No")
        if g in groupgenes:
            groupornot.append("Yes")
        else:
            groupornot.append("No")
    df = pd.DataFrame(list(zip(list(tpmdf["GeneID"].unique()),pathornot,degornot)),columns=["GeneID","is"+path,"is"+detype])
    return df

In [48]:
allclusters.head()

,cluster,GeneID,genotype,CAM_notCAM,subgenome,genotype_cluster,genotype_subgenome,genotype_cluster_subgenome,cluster_subgenome
0,1,Yucal.01G003400.v2.1,G,N,Ya,G_1,G_Ya,G_1_Ya,1_Ya
1,1,Yucal.01G004800.v2.1,G,N,Ya,G_1,G_Ya,G_1_Ya,1_Ya
2,2,Yucal.01G004900.v2.1,G,N,Ya,G_2,G_Ya,G_2_Ya,2_Ya
3,3,Yucal.01G005300.v2.1,G,N,Ya,G_3,G_Ya,G_3_Ya,3_Ya
4,4,Yucal.01G012100.v2.1,G,N,Ya,G_4,G_Ya,G_4_Ya,4_Ya


In [49]:
cinfo.head()

,cluster,genotype,n_CAM_genes,n_all_genes,gtc,ZTpeak
0,1,53,0.0,319,53_1,5.0
1,2,53,1.0,656,53_2,21.0
2,3,53,1.0,882,53_3,17.0
3,4,53,0.0,1071,53_4,9.0
4,5,53,0.0,428,53_5,21.0


In [50]:
allclusters = allclusters.merge(cinfo[["gtc","ZTpeak"]].rename(columns={"gtc":"genotype_cluster"}))
allclusters.head()

,cluster,GeneID,genotype,CAM_notCAM,subgenome,genotype_cluster,genotype_subgenome,genotype_cluster_subgenome,cluster_subgenome,ZTpeak
0,1,Yucal.01G003400.v2.1,G,N,Ya,G_1,G_Ya,G_1_Ya,1_Ya,1.0
1,1,Yucal.01G004800.v2.1,G,N,Ya,G_1,G_Ya,G_1_Ya,1_Ya,1.0
2,1,Yucal.01G014900.v2.1,G,N,Ya,G_1,G_Ya,G_1_Ya,1_Ya,1.0
3,1,Yucal.01G020100.v2.1,G,N,Ya,G_1,G_Ya,G_1_Ya,1_Ya,1.0
4,1,Yucal.01G024300.v2.1,G,N,Ya,G_1,G_Ya,G_1_Ya,1_Ya,1.0


In [62]:
# make a ZTpeak_subgenome column
allclusters["ZTpeak_subgenome"] = allclusters["ZTpeak"].astype(str)+"_"+allclusters["subgenome"]
allclusters.head()

,cluster,GeneID,genotype,CAM_notCAM,subgenome,genotype_cluster,genotype_subgenome,genotype_cluster_subgenome,cluster_subgenome,ZTpeak,ZTpeak_subgenome
0,1,Yucal.01G003400.v2.1,G,N,Ya,G_1,G_Ya,G_1_Ya,1_Ya,1.0,1.0_Ya
1,1,Yucal.01G004800.v2.1,G,N,Ya,G_1,G_Ya,G_1_Ya,1_Ya,1.0,1.0_Ya
2,1,Yucal.01G014900.v2.1,G,N,Ya,G_1,G_Ya,G_1_Ya,1_Ya,1.0,1.0_Ya
3,1,Yucal.01G020100.v2.1,G,N,Ya,G_1,G_Ya,G_1_Ya,1_Ya,1.0,1.0_Ya
4,1,Yucal.01G024300.v2.1,G,N,Ya,G_1,G_Ya,G_1_Ya,1_Ya,1.0,1.0_Ya


In [100]:
# convert ZTpeak column to str
allclusters["ZTpeak"] = allclusters["ZTpeak"].astype(str)

In [66]:
# add subgenome column to CAM annotation
sg = []
for i in list(cam["GeneID"]):
    if i.startswith("Yucal"):
        sg.append("Ya")
    else:
        sg.append("Yf")
cam["subgenome"] = sg
cam.head()

,GeneID,Orthogroup,Pathway,gene_name,gene_abbr,gene_abbr_unique,subgenome
0,Yucal.01G165600.v2.1,OG0001578,CAM-dark,beta-carbonic anhydrase,bCA1234,Ya_bCA1234_1,Ya
1,Yucal.02G112700.v2.1,OG0001578,CAM-dark,beta-carbonic anhydrase,bCA1234,Ya_bCA1234_2,Ya
2,Yucal.04G001000.v2.1,OG0004406,CAM-dark,beta-carbonic anhydrase,bCA5,Ya_bCA5_1,Ya
3,Yucal.07G000800.v2.1,OG0004406,CAM-dark,beta-carbonic anhydrase,bCA5,Ya_bCA5_2,Ya
4,Yucal.03G120900.v2.1,OG0002899,CAM-dark,NAD-dependent malate dehydrogenase (chloroplas...,NAD-MDH-cp,Ya_NAD-MDH-cp_1,Ya


In [72]:
# set up dict of TPM dataframes based on subgenome
tpmdict = {"all":ttpm,"Ya":yatpm,"Yf":yftpm}

In [101]:
def run_fisher(pathway,groupname,pathdf=cam,groupdf=allclusters):
    # get list of genes in cluster
    if "_" in groupname:
        l = len(groupname.split("_"))
        if l==3:
            gdf = allclusters[allclusters["genotype_cluster_subgenome"]==groupname]
        elif l==2:
            if groupname in list(allclusters["genotype_cluster"].unique()):
                gdf = allclusters[allclusters["genotype_cluster"]==groupname]
            elif groupname in list(allclusters["genotype_subgenome"].unique()):
                gdf = allclusters[allclusters["genotype_subgenome"]==groupname]
            elif groupname in list(allclusters["cluster_subgenome"].unique()):
                gdf = allclusters[allclusters["cluster_subgenome"]==groupname]
            elif groupname in list(allclusters["ZTpeak_subgenome"].unique()):
                gdf = allclusters[allclusters["ZTpeak_subgenome"]==groupname]
    elif groupname=="all":
        gdf = allclusters.copy()
    else:
        if type(groupname)==int:
            gdf = allclusters[allclusters["cluster"]==groupname]
        elif "." in groupname:
            gdf = allclusters[allclusters["ZTpeak"]==groupname]
        elif groupname.startswith("Y"):
            gdf = allclusters[allclusters["subgenome"]==groupname]
        else:
            gdf = allclusters[allclusters["genotype"]==groupname]
            
    groupgenes = list(gdf["GeneID"].unique())
        
    # pull out pathway genes
    if "Ya" in groupname:
        pathdf = pathdf[pathdf["subgenome"]=="Ya"]
    elif "Yf" in groupname:
        pathdf = pathdf[pathdf["subgenome"]=="Yf"]
    else:
        pathdf = pathdf
        
    if pathway=="CAM":
        pathlist = ["CAM-dark","CAM-light"]
        pathgenes = list(pathdf[pathdf["Pathway"].isin(pathlist)]["GeneID"].unique())
    else:
        pathgenes = list(pathdf[pathdf["Pathway"]==pathway]["GeneID"].unique())
        
    # set TPM dataframe
    if "Ya" in groupname:
        tpmdf = yatpm
    elif "Yf" in groupname:
        tpmdf = yftpm
    else:
        tpmdf = ttpm
        
    df = data_setup(pathgenes,groupgenes,pathway,groupname,tpmdf)
    #print(df.head())
    data = pd.crosstab(index=df["is"+pathway],columns=df["is"+groupname])
    print(data)
    odds_ratio, p_value = stats.fisher_exact(data)
    return [odds_ratio,p_value]

In [89]:
# define a function to do this for all enrichments in a given set
def test_enrichment(enrichlist,pathway,pathdf=cam,groupdf=allclusters):
    resdict = {"Group":[],"P-value":[],"Odds Ratio":[]}
    for i in enrichlist:
        pvo = run_fisher(pathway,i,pathdf,groupdf)
        resdict['Group'].append(i)
        resdict['Odds Ratio'].append(pvo[0])
        resdict['P-value'].append(pvo[1])
    #pvals = [run_fisher(pathway,i,pathdf,groupdf)[1] for i in enrichlist]
    #odds = [run_fisher(pathway,i,pathdf,groupdf)[0] for i in enrichlist]
    #res = pd.DataFrame(list(zip(enrichlist,pvals,odds)),columns=["Group","P-value","Odds Ratio"])
    res = pd.DataFrame(resdict)
    res["P-adj"] = fdrcorrection(res["P-value"])[1]
    return res

In [136]:
# create some lists of groups to test (one list?)
gtlist = list(allclusters["genotype"].unique())+list(allclusters["genotype_subgenome"].unique())

In [137]:
# subgenome/all list
sg = ["Ya","Yf","all"]

In [138]:
# genotype_cluster combinations (each & both subgenomes)
gtclist = list(allclusters["genotype_cluster"].unique())+list(allclusters["genotype_cluster_subgenome"].unique())

In [139]:
# ZT-peaks (each & both subgenomes)
ztplist = list(allclusters["ZTpeak"].unique())+list(allclusters["ZTpeak_subgenome"].unique())

In [140]:
# stick everyone together
allgroups = sg+gtlist+gtclist+ztplist

In [141]:
len(allgroups)

441

In [142]:
sg

['Ya', 'Yf', 'all']

In [143]:
# run through 'sg'
sgres = test_enrichment(sg,"CAM")

isYa     No    Yes
isCAM             
No     8664  34746
Yes       0     40
isYf     No    Yes
isCAM             
No     8196  34286
Yes       3     27
isall     No    Yes
isCAM              
No     16860  69032
Yes        3     67


In [144]:
sgres

,Group,P-value,Odds Ratio,P-adj
0,Ya,0.000217,inf,0.000613
1,Yf,0.250692,2.151432,0.250692
2,all,0.000408,5.454572,0.000613


In [145]:
# run through genotypes
gtres = test_enrichment(gtlist,"CAM")

isG       No   Yes
isCAM             
No     82121  3771
Yes       58    12
isEudy     No   Yes
isCAM              
No      77028  8864
Yes        63     7
is36      No    Yes
isCAM              
No     59744  26148
Yes       39     31
is43      No    Yes
isCAM              
No     47604  38288
Yes       32     38
is13      No   Yes
isCAM             
No     82011  3881
Yes       61     9
is45      No    Yes
isCAM              
No     69808  16084
Yes       48     22
is19      No    Yes
isCAM              
No     74877  11015
Yes       57     13
is51      No    Yes
isCAM              
No     43705  42187
Yes       27     43
is2AB     No    Yes
isCAM              
No     53481  32411
Yes       26     44
is52      No   Yes
isCAM             
No     79723  6169
Yes       61     9
is46      No    Yes
isCAM              
No     74526  11366
Yes       56     14
is70      No    Yes
isCAM              
No     63254  22638
Yes       41     29
is48      No   Yes
isCAM             
No     78940  

In [94]:
gtres[gtres["P-adj"]<0.05]

,Group,P-value,Odds Ratio,P-adj
0,G,5.113546e-05,4.505583,0.000614
2,36,1.850985e-02,1.816155,0.046275
4,13,4.237065e-03,3.117749,0.018159
5,45,1.302183e-02,1.989265,0.036732
8,2AB,3.204128e-05,2.792457,0.000481
11,70,6.263404e-03,1.976351,0.023488
13,55,1.800725e-02,1.795500,0.046275
15,61,5.780650e-06,3.016014,0.000116
17,56,3.864996e-03,2.246495,0.017838
18,1AB,4.811459e-06,3.091385,0.000116


In [146]:
# run through individual clusters (genotype-cluster)
gtcres = test_enrichment(gtclist,"CAM")

isG_1     No  Yes
isCAM            
No     84918  974
Yes       65    5
isG_2     No  Yes
isCAM            
No     84982  910
Yes       66    4
isG_3     No  Yes
isCAM            
No     85351  541
Yes       69    1
isG_4     No  Yes
isCAM            
No     85270  622
Yes       70    0
isG_5     No  Yes
isCAM            
No     85739  153
Yes       70    0
isG_6     No  Yes
isCAM            
No     85321  571
Yes       68    2
isEudy_1     No   Yes
isCAM                
No        84096  1796
Yes          70     0
isEudy_2     No   Yes
isCAM                
No        84238  1654
Yes          69     1
isEudy_3     No   Yes
isCAM                
No        84664  1228
Yes          69     1
isEudy_4     No   Yes
isCAM                
No        83501  2391
Yes          66     4
isEudy_5     No  Yes
isCAM               
No        84917  975
Yes          69    1
isEudy_6     No  Yes
isCAM               
No        85072  820
Yes          70    0
is36_1     No   Yes
isCAM              
No      

is56_2     No   Yes
isCAM              
No      82292  3600
Yes        68     2
is56_3     No   Yes
isCAM              
No      82705  3187
Yes        59    11
is56_4     No   Yes
isCAM              
No      83360  2532
Yes        67     3
is56_5     No  Yes
isCAM             
No      84979  913
Yes        68    2
is56_6     No  Yes
isCAM             
No      85184  708
Yes        70    0
is1AB_1     No   Yes
isCAM               
No       84484  1408
Yes         68     2
is1AB_2     No   Yes
isCAM               
No       81059  4833
Yes         57    13
is1AB_3     No   Yes
isCAM               
No       81696  4196
Yes         63     7
is1AB_4     No   Yes
isCAM               
No       83195  2697
Yes         68     2
is1AB_5     No   Yes
isCAM               
No       82062  3830
Yes         66     4
is1AB_6     No   Yes
isCAM               
No       83624  2268
Yes         65     5
is37_1     No   Yes
isCAM              
No      80704  5188
Yes        70     0
is37_2     No   Yes
isCA

is19_3_Ya     No   Yes
isCAM                 
No         41945  1465
Yes           37     3
is19_3_Yf     No   Yes
isCAM                 
No         41104  1378
Yes           28     2
is19_4_Ya     No   Yes
isCAM                 
No         42356  1054
Yes           39     1
is19_4_Yf     No  Yes
isCAM                
No         41488  994
Yes           28    2
is19_5_Ya     No   Yes
isCAM                 
No         42400  1010
Yes           39     1
is19_5_Yf     No  Yes
isCAM                
No         41512  970
Yes           29    1
is19_6_Ya     No  Yes
isCAM                
No         42627  783
Yes           40    0
is19_6_Yf     No  Yes
isCAM                
No         41767  715
Yes           30    0
is51_1_Ya     No   Yes
isCAM                 
No         37487  5923
Yes           37     3
is51_1_Yf     No   Yes
isCAM                 
No         36952  5530
Yes           30     0
is51_2_Ya     No   Yes
isCAM                 
No         38368  5042
Yes           33     7
is51

is55_6_Yf     No   Yes
isCAM                 
No         39218  3264
Yes           28     2
is18_1_Ya     No   Yes
isCAM                 
No         40866  2544
Yes           38     2
is18_1_Yf     No   Yes
isCAM                 
No         40144  2338
Yes           29     1
is18_2_Ya     No   Yes
isCAM                 
No         39936  3474
Yes           34     6
is18_2_Yf     No   Yes
isCAM                 
No         39191  3291
Yes           26     4
is18_3_Ya     No   Yes
isCAM                 
No         42399  1011
Yes           39     1
is18_3_Yf     No  Yes
isCAM                
No         41536  946
Yes           29    1
is18_4_Ya     No   Yes
isCAM                 
No         41098  2312
Yes           35     5
is18_4_Yf     No   Yes
isCAM                 
No         40173  2309
Yes           27     3
is18_5_Ya     No   Yes
isCAM                 
No         40873  2537
Yes           37     3
is18_5_Yf     No   Yes
isCAM                 
No         40061  2421
Yes           2

In [97]:
gtcres[gtcres["P-adj"]<0.05]

,Group,P-value,Odds Ratio,P-adj
0,G_1,0.001246,6.706523,0.032032
25,13_2,0.000498,8.265346,0.019235
47,51_6,0.000033,5.116130,0.005890
91,61_2,0.000014,3.451974,0.005005
94,61_5,0.001044,3.178655,0.028898
104,56_3,0.000053,4.838273,0.006371
109,1AB_2,0.000131,3.825189,0.007874
116,37_3,0.000121,3.152122,0.007874
215,51_6_Yf,0.000534,6.892233,0.019235
219,2AB_2_Yf,0.001979,5.290167,0.044988


In [147]:
# now run ZTpeaks
ztpres = test_enrichment(ztplist,"CAM")

is1.0     No    Yes
isCAM              
No     58799  27093
Yes       32     38
is9.0     No    Yes
isCAM              
No     51800  34092
Yes       28     42
is5.0     No    Yes
isCAM              
No     55255  30637
Yes       29     41
is21.0     No    Yes
isCAM               
No      57251  28641
Yes        44     26
is13.0     No    Yes
isCAM               
No      62734  23158
Yes        51     19
is17.0     No    Yes
isCAM               
No      57804  28088
Yes        45     25
is1.0_Ya     No    Yes
isCAM                 
No        29887  13523
Yes          18     22
is1.0_Yf     No    Yes
isCAM                 
No        28912  13570
Yes          14     16
is9.0_Ya     No    Yes
isCAM                 
No        26347  17063
Yes          15     25
is9.0_Yf     No    Yes
isCAM                 
No        25453  17029
Yes          13     17
is5.0_Ya     No    Yes
isCAM                 
No        28080  15330
Yes          14     26
is5.0_Yf     No    Yes
isCAM                 
No

In [104]:
ztpres[ztpres["P-adj"]<0.05]

,Group,P-value,Odds Ratio,P-adj
0,1.0,0.000088,2.577190,0.001061
1,9.0,0.000832,2.279127,0.003746
2,5.0,0.000137,2.549830,0.001061
6,1.0_Ya,0.001850,2.701217,0.006660
7,1.0_Yf,0.017399,2.434951,0.044739
8,9.0_Ya,0.003332,2.573502,0.009997
10,5.0_Ya,0.000177,3.401733,0.001061


In [148]:
# append all results dataframes together
allres = pd.concat([sgres,ztpres,gtres,gtcres])

In [149]:
# out of curiosity: do FDR pvals change when the n is much higher?
allres["P-adj_highN"] = fdrcorrection(allres["P-value"])[1]

In [150]:
allres.head()

,Group,P-value,Odds Ratio,P-adj,P-adj_highN
0,Ya,0.000217,inf,0.000613,0.005989
1,Yf,0.250692,2.151432,0.250692,0.636521
2,all,0.000408,5.454572,0.000613,0.009478
0,1.0,0.000088,2.577190,0.001061,0.003894
1,9.0,0.000832,2.279127,0.003746,0.015295


In [151]:
# yes - high n is more conservative (at least in most cases)
# get significant results based on P-adj_highN column
sigres = allres[allres["P-adj_highN"]<0.05]

In [152]:
len(allres[allres["P-adj"]<0.05].index)

49

In [153]:
len(sigres.index)

50

In [154]:
# create a subgenome column in sigres
subg = []
for i in list(sigres["Group"]):
    if "Ya" in i:
        subg.append("Ya")
    elif "Yf" in i:
        subg.append("Yf")
    else:
        subg.append("both")

In [155]:
sigres["subgenome"] = subg
sigres.head()

/tmp/ipykernel_140/1116880248.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sigres["subgenome"] = subg


,Group,P-value,Odds Ratio,P-adj,P-adj_highN,subgenome
0,Ya,0.000217,inf,0.000613,0.005989,Ya
2,all,0.000408,5.454572,0.000613,0.009478,both
0,1.0,0.000088,2.577190,0.001061,0.003894,both
1,9.0,0.000832,2.279127,0.003746,0.015295,both
2,5.0,0.000137,2.549830,0.001061,0.004661,both


In [156]:
sigres.groupby("subgenome").count().reset_index()[["subgenome","Group"]]

,subgenome,Group
0,Ya,22
1,Yf,4
2,both,24


In [157]:
sigres[sigres["subgenome"]=="Yf"]

,Group,P-value,Odds Ratio,P-adj,P-adj_highN,subgenome
57,1AB_Yf,0.002853,3.127557,0.017115,0.036999,Yf
215,51_6_Yf,0.000534,6.892233,0.019235,0.011220,Yf
219,2AB_2_Yf,0.001979,5.290167,0.044988,0.028444,Yf
329,56_3_Yf,0.000627,6.677919,0.020233,0.012566,Yf


In [158]:
# create a "group type" column for sigres (for easier filtering)
grptype = []
for i in list(sigres["Group"]):
    if i in sg:
        grptype.append("overall_TSgenes")
    elif i in gtlist:
        grptype.append("genotype_TSgenes")
    elif i in gtclist:
        grptype.append("individual_cluster")
    elif i in ztplist:
        grptype.append("ZTpeak_clustergroup")
sigres["group_type"] = grptype
sigres.head()

/tmp/ipykernel_140/3525569891.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sigres["group_type"] = grptype


,Group,P-value,Odds Ratio,P-adj,P-adj_highN,subgenome,group_type
0,Ya,0.000217,inf,0.000613,0.005989,Ya,overall_TSgenes
2,all,0.000408,5.454572,0.000613,0.009478,both,overall_TSgenes
0,1.0,0.000088,2.577190,0.001061,0.003894,both,ZTpeak_clustergroup
1,9.0,0.000832,2.279127,0.003746,0.015295,both,ZTpeak_clustergroup
2,5.0,0.000137,2.549830,0.001061,0.004661,both,ZTpeak_clustergroup


In [159]:
sigres[sigres["group_type"]=="overall_TSgenes"]

,Group,P-value,Odds Ratio,P-adj,P-adj_highN,subgenome,group_type
0,Ya,0.000217,inf,0.000613,0.005989,Ya,overall_TSgenes
2,all,0.000408,5.454572,0.000613,0.009478,both,overall_TSgenes


In [160]:
sigres[sigres["group_type"]=="ZTpeak_clustergroup"]

,Group,P-value,Odds Ratio,P-adj,P-adj_highN,subgenome,group_type
0,1.0,0.000088,2.577190,0.001061,0.003894,both,ZTpeak_clustergroup
1,9.0,0.000832,2.279127,0.003746,0.015295,both,ZTpeak_clustergroup
2,5.0,0.000137,2.549830,0.001061,0.004661,both,ZTpeak_clustergroup
6,1.0_Ya,0.001850,2.701217,0.006660,0.028131,Ya,ZTpeak_clustergroup
8,9.0_Ya,0.003332,2.573502,0.009997,0.039639,Ya,ZTpeak_clustergroup
10,5.0_Ya,0.000177,3.401733,0.001061,0.005571,Ya,ZTpeak_clustergroup


In [161]:
sigres[sigres["group_type"]=="genotype_TSgenes"]

,Group,P-value,Odds Ratio,P-adj,P-adj_highN,subgenome,group_type
0,G,5.113546e-05,4.505583,0.000614,0.002927,both,genotype_TSgenes
4,13,4.237065e-03,3.117749,0.018159,0.042415,both,genotype_TSgenes
8,2AB,3.204128e-05,2.792457,0.000481,0.002405,both,genotype_TSgenes
15,61,5.780650e-06,3.016014,0.000116,0.000850,both,genotype_TSgenes
17,56,3.864996e-03,2.246495,0.017838,0.039639,both,genotype_TSgenes
18,1AB,4.811459e-06,3.091385,0.000116,0.000850,both,genotype_TSgenes
20,G_Ya,3.858391e-04,5.244937,0.003858,0.009478,Ya,genotype_TSgenes
24,36_Ya,5.285763e-03,2.486015,0.021143,0.048450,Ya,genotype_TSgenes
36,2AB_Ya,8.928607e-04,2.978714,0.007034,0.015750,Ya,genotype_TSgenes
42,70_Ya,3.826409e-03,2.531434,0.017838,0.039639,Ya,genotype_TSgenes


In [162]:
sigres[sigres["group_type"]=="individual_cluster"]

,Group,P-value,Odds Ratio,P-adj,P-adj_highN,subgenome,group_type
0,G_1,0.001246,6.706523,0.032032,0.019620,both,individual_cluster
12,36_1,0.004385,2.598256,0.065095,0.042415,both,individual_cluster
25,13_2,0.000498,8.265346,0.019235,0.010980,both,individual_cluster
30,45_1,0.003410,4.419913,0.062112,0.039639,both,individual_cluster
47,51_6,0.000033,5.116130,0.005890,0.002405,both,individual_cluster
49,2AB_2,0.004520,3.085263,0.065095,0.042415,both,individual_cluster
70,70_5,0.005383,2.998584,0.074539,0.048450,both,individual_cluster
76,48_5,0.002632,5.628670,0.055732,0.036270,both,individual_cluster
79,55_2,0.003744,2.823665,0.062112,0.039639,both,individual_cluster
91,61_2,0.000014,3.451974,0.005005,0.001533,both,individual_cluster


In [163]:
# save both allres and sigres
allres.to_csv("./camenrich_tsgenes_allp.txt",sep="\t",header=True,index=False)
sigres.to_csv("./camenrich_tsgenes_sigp.txt",sep="\t",header=True,index=False)

In [165]:
cam["Pathway"].unique()

array(['CAM-dark', 'CAM-light'], dtype=object)

In [166]:
# repeat for CAM-dark and CAM-light
darkres = test_enrichment(allgroups,"CAM-dark")

isYa          No    Yes
isCAM-dark             
No          8664  34760
Yes            0     26
isYf          No    Yes
isCAM-dark             
No          8197  34294
Yes            2     19
isall          No    Yes
isCAM-dark              
No          16861  69054
Yes             2     45
isG            No   Yes
isCAM-dark             
No          82140  3775
Yes            39     8
isEudy         No   Yes
isCAM-dark             
No          77046  8869
Yes            45     2
is36           No    Yes
isCAM-dark              
No          59755  26160
Yes            28     19
is43           No    Yes
isCAM-dark              
No          47614  38301
Yes            22     25
is13           No   Yes
isCAM-dark             
No          82030  3885
Yes            42     5
is45           No    Yes
isCAM-dark              
No          69821  16094
Yes            35     12
is19           No    Yes
isCAM-dark              
No          74898  11017
Yes            36     11
is51           No   

is43_5         No   Yes
isCAM-dark             
No          79627  6288
Yes            41     6
is43_6         No   Yes
isCAM-dark             
No          81945  3970
Yes            42     5
is13_1         No  Yes
isCAM-dark            
No          85014  901
Yes            46    1
is13_2         No  Yes
isCAM-dark            
No          85122  793
Yes            43    4
is13_3         No  Yes
isCAM-dark            
No          85239  676
Yes            47    0
is13_4         No   Yes
isCAM-dark             
No          84889  1026
Yes            47     0
is13_5         No  Yes
isCAM-dark            
No          85706  209
Yes            47    0
is13_6         No  Yes
isCAM-dark            
No          85635  280
Yes            47    0
is45_1         No   Yes
isCAM-dark             
No          84126  1789
Yes            46     1
is45_2         No   Yes
isCAM-dark             
No          82036  3879
Yes            42     5
is45_3         No   Yes
isCAM-dark             
No          

is1AB_2        No   Yes
isCAM-dark             
No          81077  4838
Yes            39     8
is1AB_3        No   Yes
isCAM-dark             
No          81715  4200
Yes            44     3
is1AB_4        No   Yes
isCAM-dark             
No          83217  2698
Yes            46     1
is1AB_5        No   Yes
isCAM-dark             
No          82084  3831
Yes            44     3
is1AB_6        No   Yes
isCAM-dark             
No          83645  2270
Yes            44     3
is37_1         No   Yes
isCAM-dark             
No          80727  5188
Yes            47     0
is37_2         No   Yes
isCAM-dark             
No          79371  6544
Yes            43     4
is37_3         No   Yes
isCAM-dark             
No          77410  8505
Yes            35    12
is37_4         No    Yes
isCAM-dark              
No          75767  10148
Yes            37     10
is37_5         No   Yes
isCAM-dark             
No          76712  9203
Yes            43     4
is37_6         No   Yes
isCAM-dark  

is19_3_Ya      No   Yes
isCAM-dark             
No          41958  1466
Yes            24     2
is19_3_Yf      No   Yes
isCAM-dark             
No          41112  1379
Yes            20     1
is19_4_Ya      No   Yes
isCAM-dark             
No          42370  1054
Yes            25     1
is19_4_Yf      No  Yes
isCAM-dark            
No          41497  994
Yes            19    2
is19_5_Ya      No   Yes
isCAM-dark             
No          42414  1010
Yes            25     1
is19_5_Yf      No  Yes
isCAM-dark            
No          41521  970
Yes            20    1
is19_6_Ya      No  Yes
isCAM-dark            
No          42641  783
Yes            26    0
is19_6_Yf      No  Yes
isCAM-dark            
No          41776  715
Yes            21    0
is51_1_Ya      No   Yes
isCAM-dark             
No          37500  5924
Yes            24     2
is51_1_Yf      No   Yes
isCAM-dark             
No          36961  5530
Yes            21     0
is51_2_Ya      No   Yes
isCAM-dark             
No      

is55_4_Yf      No   Yes
isCAM-dark             
No          41027  1464
Yes            20     1
is55_5_Ya      No   Yes
isCAM-dark             
No          39311  4113
Yes            22     4
is55_5_Yf      No   Yes
isCAM-dark             
No          38666  3825
Yes            21     0
is55_6_Ya      No   Yes
isCAM-dark             
No          39959  3465
Yes            23     3
is55_6_Yf      No   Yes
isCAM-dark             
No          39227  3264
Yes            19     2
is18_1_Ya      No   Yes
isCAM-dark             
No          40879  2545
Yes            25     1
is18_1_Yf      No   Yes
isCAM-dark             
No          40152  2339
Yes            21     0
is18_2_Ya      No   Yes
isCAM-dark             
No          39949  3475
Yes            21     5
is18_2_Yf      No   Yes
isCAM-dark             
No          39199  3292
Yes            18     3
is18_3_Ya      No   Yes
isCAM-dark             
No          42413  1011
Yes            25     1
is18_3_Yf      No  Yes
isCAM-dark       

is9.0_Yf       No    Yes
isCAM-dark              
No          25458  17033
Yes             8     13
is5.0_Ya       No    Yes
isCAM-dark              
No          28081  15343
Yes            13     13
is5.0_Yf       No    Yes
isCAM-dark              
No          27178  15313
Yes            12      9
is21.0_Ya      No    Yes
isCAM-dark              
No          29131  14293
Yes            17      9
is21.0_Yf      No    Yes
isCAM-dark              
No          28137  14354
Yes            10     11
is13.0_Ya      No    Yes
isCAM-dark              
No          31576  11848
Yes            17      9
is13.0_Yf      No    Yes
isCAM-dark              
No          31175  11316
Yes            17      4
is17.0_Ya      No    Yes
isCAM-dark              
No          29131  14293
Yes            12     14
is17.0_Yf      No    Yes
isCAM-dark              
No          28691  13800
Yes            15      6


In [167]:
lightres = test_enrichment(allgroups,"CAM-light")

isYa           No    Yes
isCAM-light             
No           8664  34772
Yes             0     14
isYf           No    Yes
isCAM-light             
No           8198  34305
Yes             1      8
isall           No    Yes
isCAM-light              
No           16862  69077
Yes              1     22
isG             No   Yes
isCAM-light             
No           82160  3779
Yes             19     4
isEudy          No   Yes
isCAM-light             
No           77073  8866
Yes             18     5
is36            No    Yes
isCAM-light              
No           59772  26167
Yes             11     12
is43            No    Yes
isCAM-light              
No           47626  38313
Yes             10     13
is13            No   Yes
isCAM-light             
No           82053  3886
Yes             19     4
is45            No    Yes
isCAM-light              
No           69843  16096
Yes             13     10
is19            No    Yes
isCAM-light              
No           74913  11026
Yes   

is43_1          No   Yes
isCAM-light             
No           82627  3312
Yes             21     2
is43_2          No   Yes
isCAM-light             
No           77999  7940
Yes             21     2
is43_3          No    Yes
isCAM-light              
No           74870  11069
Yes             22      1
is43_4          No   Yes
isCAM-light             
No           80211  5728
Yes             20     3
is43_5          No   Yes
isCAM-light             
No           79648  6291
Yes             20     3
is43_6          No   Yes
isCAM-light             
No           81966  3973
Yes             21     2
is13_1          No  Yes
isCAM-light            
No           85037  902
Yes             23    0
is13_2          No  Yes
isCAM-light            
No           85143  796
Yes             22    1
is13_3          No  Yes
isCAM-light            
No           85263  676
Yes             23    0
is13_4          No   Yes
isCAM-light             
No           84916  1023
Yes             20     3
is13_5  

is53_6          No  Yes
isCAM-light            
No           85162  777
Yes             23    0
is56_1          No   Yes
isCAM-light             
No           83896  2043
Yes             22     1
is56_2          No   Yes
isCAM-light             
No           82337  3602
Yes             23     0
is56_3          No   Yes
isCAM-light             
No           82746  3193
Yes             18     5
is56_4          No   Yes
isCAM-light             
No           83407  2532
Yes             20     3
is56_5          No  Yes
isCAM-light            
No           85024  915
Yes             23    0
is56_6          No  Yes
isCAM-light            
No           85231  708
Yes             23    0
is1AB_1         No   Yes
isCAM-light             
No           84529  1410
Yes             23     0
is1AB_2         No   Yes
isCAM-light             
No           81098  4841
Yes             18     5
is1AB_3         No   Yes
isCAM-light             
No           81740  4199
Yes             19     4
is1AB_4     

is45_3_Yf       No  Yes
isCAM-light            
No           41754  749
Yes              9    0
is45_4_Ya       No   Yes
isCAM-light             
No           41698  1738
Yes             13     1
is45_4_Yf       No   Yes
isCAM-light             
No           40868  1635
Yes              8     1
is45_5_Ya       No  Yes
isCAM-light            
No           42494  942
Yes             13    1
is45_5_Yf       No  Yes
isCAM-light            
No           41651  852
Yes              9    0
is45_6_Ya       No   Yes
isCAM-light             
No           41532  1904
Yes             14     0
is45_6_Yf       No   Yes
isCAM-light             
No           40687  1816
Yes              9     0
is19_1_Ya       No  Yes
isCAM-light            
No           42677  759
Yes             14    0
is19_1_Yf       No  Yes
isCAM-light            
No           41889  614
Yes              9    0
is19_2_Ya       No  Yes
isCAM-light            
No           42755  681
Yes             14    0
is19_2_Yf       No  Yes


is48_3_Yf       No  Yes
isCAM-light            
No           41543  960
Yes              9    0
is48_4_Ya       No  Yes
isCAM-light            
No           42695  741
Yes             12    2
is48_4_Yf       No  Yes
isCAM-light            
No           41815  688
Yes              9    0
is48_5_Ya       No  Yes
isCAM-light            
No           42854  582
Yes             13    1
is48_5_Yf       No  Yes
isCAM-light            
No           41924  579
Yes              8    1
is48_6_Ya       No  Yes
isCAM-light            
No           43145  291
Yes             14    0
is48_6_Yf       No  Yes
isCAM-light            
No           42193  310
Yes              9    0
is55_1_Ya       No   Yes
isCAM-light             
No           40782  2654
Yes             14     0
is55_1_Yf       No   Yes
isCAM-light             
No           40166  2337
Yes              8     1
is55_2_Ya       No   Yes
isCAM-light             
No           40716  2720
Yes              9     5
is55_2_Yf       No   Yes
isC

is37_3_Yf       No   Yes
isCAM-light             
No           38298  4205
Yes              8     1
is37_4_Ya       No   Yes
isCAM-light             
No           38246  5190
Yes             12     2
is37_4_Yf       No   Yes
isCAM-light             
No           37540  4963
Yes              6     3
is37_5_Ya       No   Yes
isCAM-light             
No           38783  4653
Yes             14     0
is37_5_Yf       No   Yes
isCAM-light             
No           37949  4554
Yes              9     0
is37_6_Ya       No   Yes
isCAM-light             
No           39920  3516
Yes             12     2
is37_6_Yf       No   Yes
isCAM-light             
No           39057  3446
Yes              8     1
is1.0           No    Yes
isCAM-light              
No           58821  27118
Yes             10     13
is9.0           No    Yes
isCAM-light              
No           51820  34119
Yes              8     15
is5.0           No    Yes
isCAM-light              
No           55280  30659
Yes           

In [168]:
darkres.head()

,Group,P-value,Odds Ratio,P-adj
0,Ya,0.005350,inf,0.117970
1,Yf,0.405184,2.270703,0.926694
2,all,0.005007,5.493853,0.117970
3,G,0.000945,4.463372,0.084081
4,Eudy,0.230351,0.386094,0.751373


In [170]:
allres.drop("P-adj_highN",axis=1,inplace=True)

In [171]:
allres["CAM_set"] = "all"
darkres["CAM_set"] = "dark"
lightres["CAM_set"] = "light"

In [172]:
camenrich = pd.concat([allres,darkres,lightres])
len(camenrich.index)

1323

In [174]:
# re-calculate FDR for everything all together
camenrich["P-adj_highN"] = fdrcorrection(camenrich["P-value"])[1]
camenrich.head()

,Group,P-value,Odds Ratio,P-adj,CAM_set,P-adj_highN
0,Ya,0.000217,inf,0.000613,all,0.011533
1,Yf,0.250692,2.151432,0.250692,all,0.710114
2,all,0.000408,5.454572,0.000613,all,0.017427
0,1.0,0.000088,2.577190,0.001061,all,0.007789
1,9.0,0.000832,2.279127,0.003746,all,0.028478


In [178]:
# add other useful columns: subgenome & group type
subg = []
grptype = []
for i in list(camenrich["Group"]):
    if "Ya" in i:
        subg.append("Ya")
    elif "Yf" in i:
        subg.append("Yf")
    else:
        subg.append("both")
    
    if i in sg:
        grptype.append("overall_TSgenes")
    elif i in gtlist:
        grptype.append("genotype_TSgenes")
    elif i in ztplist:
        grptype.append("ZTpeak_clustergroup")
    elif i in gtclist:
        grptype.append("individual_cluster")
        
camenrich["subgenome"] = subg
camenrich["group_type"] = grptype

In [179]:
camsig = camenrich[camenrich["P-adj_highN"]<0.05]
len(camsig.index)

57

In [180]:
camsig.tail()

,Group,P-value,Odds Ratio,P-adj,CAM_set,P-adj_highN,subgenome,group_type
347,55_5_Ya,0.001106,7.174349,0.037522,light,0.032519,Ya,individual_cluster
365,61_2_Ya,0.000033,10.877223,0.003873,light,0.004647,Ya,individual_cluster
401,1AB_2_Ya,0.000839,9.043136,0.030851,light,0.028478,Ya,individual_cluster
425,5.0,0.000005,8.564532,0.002348,light,0.001912,both,ZTpeak_clustergroup
433,5.0_Ya,0.000013,23.802972,0.002775,light,0.003066,Ya,ZTpeak_clustergroup


In [183]:
# save both camenrich and camsig (overwrite previous files)
camenrich.to_csv("./camenrich_tsgenes_allp.txt",sep="\t",header=True,index=False)
camsig.to_csv("./camenrich_tsgenes_sigp.txt",sep="\t",header=True,index=False)

In [181]:
camsig[camsig["CAM_set"]=="light"]

,Group,P-value,Odds Ratio,P-adj,CAM_set,P-adj_highN,subgenome,group_type
16,55,0.001939,3.875505,0.050302,light,0.048096,both,genotype_TSgenes
17,18,0.001532,3.988692,0.042213,light,0.041352,both,genotype_TSgenes
18,61,0.000140,5.167596,0.008719,light,0.009290,both,genotype_TSgenes
21,1AB,0.000405,4.503070,0.017875,light,0.017427,both,genotype_TSgenes
53,61_Ya,0.000035,13.430105,0.003873,light,0.004647,Ya,genotype_TSgenes
59,1AB_Ya,0.000158,8.409182,0.008719,light,0.009964,Ya,genotype_TSgenes
93,45_1,0.000096,13.095861,0.007043,light,0.007923,both,individual_cluster
142,55_2,0.000356,6.624542,0.017425,light,0.017427,both,individual_cluster
145,55_5,0.000726,5.243593,0.029121,light,0.025973,both,individual_cluster
154,61_2,0.000079,6.191591,0.007006,light,0.007634,both,individual_cluster


In [182]:
camsig[camsig["CAM_set"]=="dark"]

,Group,P-value,Odds Ratio,P-adj,CAM_set,P-adj_highN,subgenome,group_type
3,G,0.000945,4.463372,0.084081,dark,0.029331,both,genotype_TSgenes
11,2AB,0.000403,2.911152,0.084081,dark,0.017427,both,genotype_TSgenes
53,61_Ya,0.002056,3.581475,0.098278,dark,0.048567,Ya,genotype_TSgenes
60,1AB_Yf,0.002107,3.931227,0.098278,dark,0.048915,Yf,genotype_TSgenes
64,G_2,0.001575,8.689497,0.098278,dark,0.041686,both,individual_cluster
88,13_2,0.000953,9.985278,0.084081,dark,0.029331,both,individual_cluster
110,51_6,0.000218,5.624706,0.084081,dark,0.011533,both,individual_cluster
179,37_3,0.001692,3.120585,0.098278,dark,0.043900,both,individual_cluster
278,51_6_Yf,0.000675,8.611181,0.084081,dark,0.024795,Yf,individual_cluster


In [184]:
# make a nice summary table of genotype results (with FDR p-values)
gttab = camsig[camsig["group_type"]=="genotype_TSgenes"]

In [187]:
gttab.head()

,Genotype,P-value,Odds Ratio,P-adj,CAM_set,P-adj_highN,subgenome,group_type
0,G,0.000051,4.505583,0.000614,all,0.005853,both,genotype_TSgenes
8,2AB,0.000032,2.792457,0.000481,all,0.004647,both,genotype_TSgenes
15,61,0.000006,3.016014,0.000116,all,0.001912,both,genotype_TSgenes
18,1AB,0.000005,3.091385,0.000116,all,0.001912,both,genotype_TSgenes
20,G_Ya,0.000386,5.244937,0.003858,all,0.017427,Ya,genotype_TSgenes


In [186]:
gttab.rename(columns={"Group":"Genotype"},inplace=True)

/tmp/ipykernel_140/1404136697.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gttab.rename(columns={"Group":"Genotype"},inplace=True)


In [188]:
gttab.drop(["P-value","Odds Ratio","P-adj","group_type"],axis=1,inplace=True)
gttab.rename(columns={"P-adj-highN":"P-adj"},inplace=True)

/tmp/ipykernel_140/231369202.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gttab.drop(["P-value","Odds Ratio","P-adj","group_type"],axis=1,inplace=True)
/tmp/ipykernel_140/231369202.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gttab.rename(columns={"P-adj-highN":"P-adj"},inplace=True)


In [189]:
gttab.head()

,Genotype,CAM_set,P-adj_highN,subgenome
0,G,all,0.005853,both
8,2AB,all,0.004647,both
15,61,all,0.001912,both
18,1AB,all,0.001912,both
20,G_Ya,all,0.017427,Ya


In [190]:
for i in list(gttab["Genotype"]):
    if "_" in i:
        gttab["Genotype"].replace(i,i.split("_")[0],inplace=True)

/tmp/ipykernel_140/358127644.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gttab["Genotype"].replace(i,i.split("_")[0],inplace=True)


In [191]:
gttab.head()

,Genotype,CAM_set,P-adj_highN,subgenome
0,G,all,0.005853,both
8,2AB,all,0.004647,both
15,61,all,0.001912,both
18,1AB,all,0.001912,both
20,G,all,0.017427,Ya


In [192]:
gttab.rename(columns={"P-adj_highN":"P-adj"},inplace=True)
gttab.head()

/tmp/ipykernel_140/3219333157.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gttab.rename(columns={"P-adj_highN":"P-adj"},inplace=True)


,Genotype,CAM_set,P-adj,subgenome
0,G,all,0.005853,both
8,2AB,all,0.004647,both
15,61,all,0.001912,both
18,1AB,all,0.001912,both
20,G,all,0.017427,Ya


In [193]:
gttab = gttab[["Genotype","subgenome","CAM_set","P-adj"]]
gttab.head()

,Genotype,subgenome,CAM_set,P-adj
0,G,both,all,0.005853
8,2AB,both,all,0.004647
15,61,both,all,0.001912
18,1AB,both,all,0.001912
20,G,Ya,all,0.017427


In [194]:
gttab.sort_values(by="Genotype",ascending=True,inplace=True)

In [195]:
gttab.head()

,Genotype,subgenome,CAM_set,P-adj
17,18,both,light,0.041352
59,1AB,Ya,light,0.009964
18,1AB,both,all,0.001912
21,1AB,both,light,0.017427
56,1AB,Ya,all,0.029331


In [196]:
# save gttab
gttab.to_csv("./summarytable_genotype_camenrich_sig.csv",sep=",",header=True,index=False)

In [2]:
# Dec. 15
# reload camsig
camsig = pd.read_csv("./camenrich_tsgenes_sigp.txt",sep="\t",header="infer")
camsig.head()

,Group,P-value,Odds Ratio,P-adj,CAM_set,P-adj_highN,subgenome,group_type
0,Ya,0.000217,inf,0.000613,all,0.011533,Ya,overall_TSgenes
1,all,0.000408,5.454572,0.000613,all,0.017427,both,overall_TSgenes
2,1.0,0.000088,2.577190,0.001061,all,0.007789,both,ZTpeak_clustergroup
3,9.0,0.000832,2.279127,0.003746,all,0.028478,both,ZTpeak_clustergroup
4,5.0,0.000137,2.549830,0.001061,all,0.009290,both,ZTpeak_clustergroup


In [5]:
camsig["group_type"].unique()

array(['overall_TSgenes', 'ZTpeak_clustergroup', 'genotype_TSgenes',
       'individual_cluster'], dtype=object)

In [17]:
camsig[camsig["group_type"]=="overall_TSgenes"]

,Group,P-value,Odds Ratio,P-adj,CAM_set,P-adj_highN,subgenome,group_type
0,Ya,0.000217,inf,0.000613,all,0.011533,Ya,overall_TSgenes
1,all,0.000408,5.454572,0.000613,all,0.017427,both,overall_TSgenes


In [7]:
# make a summary table of the individual clusters
cres = camsig[camsig["group_type"]=="individual_cluster"]

In [8]:
cres.rename(columns={"Group":"ClusterID"},inplace=True)

/tmp/ipykernel_5985/4172504066.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cres.rename(columns={"Group":"ClusterID"},inplace=True)


In [9]:
cres.drop(["P-value","Odds Ratio","P-adj","group_type"],axis=1,inplace=True)
cres.rename(columns={"P-adj-highN":"P-adj"},inplace=True)

/tmp/ipykernel_5985/797454565.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cres.drop(["P-value","Odds Ratio","P-adj","group_type"],axis=1,inplace=True)
/tmp/ipykernel_5985/797454565.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cres.rename(columns={"P-adj-highN":"P-adj"},inplace=True)


In [10]:
cres.head()

,ClusterID,CAM_set,P-adj_highN,subgenome
15,G_1,all,0.035065,both
16,13_2,all,0.020587,both
17,51_6,all,0.004647,both
18,61_2,all,0.003066,both
19,61_5,all,0.031377,both


In [11]:
# split ClusterID column into Genotype and Cluster Number
gt = []
cnum = []
for i in list(cres["ClusterID"]):
    gt.append(i.split("_")[0])
    cnum.append(i.split("_")[1])
cres["Genotype"] = gt
cres["Cluster"] = cnum
cres.head()

/tmp/ipykernel_5985/3965179623.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cres["Genotype"] = gt
/tmp/ipykernel_5985/3965179623.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cres["Cluster"] = cnum


,ClusterID,CAM_set,P-adj_highN,subgenome,Genotype,Cluster
15,G_1,all,0.035065,both,G,1
16,13_2,all,0.020587,both,13,2
17,51_6,all,0.004647,both,51,6
18,61_2,all,0.003066,both,61,2
19,61_5,all,0.031377,both,61,5


In [12]:
cres = cres[["Genotype","Cluster","CAM_set","P-adj_highN"]]

In [13]:
cres.rename(columns={"P-adj_highN":"P-adj"},inplace=True)

In [14]:
cres.head()

,Genotype,Cluster,CAM_set,P-adj
15,G,1,all,0.035065
16,13,2,all,0.020587
17,51,6,all,0.004647
18,61,2,all,0.003066
19,61,5,all,0.031377


In [15]:
cres.to_csv("./summarytable_camenrich_sigclusters.csv",sep=",",header=True,index=False)